In [11]:
import csv
import json

with open("appointments.csv", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    print(reader.fieldnames)
    raw_rows = list(reader)

print("Rows read:", len(raw_rows))

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020


In [12]:
def parse_row(row):
    parsed = row.copy()
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed


accepted = []
rejected = []

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

Accepted: 988
Rejected: 32


In [13]:
import pandas as pd

df = pd.read_csv("appointments.csv")

print("Raw shape:", df.shape)
df.info()
print("\nMissing values by column:")
print(df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())
print("Raw booking-channel spellings:", df["booking_channel"].nunique())

# (a) Remove exact duplicates.
df = df.drop_duplicates().copy()
print("\nRows after dropping duplicates:", len(df))

# (b) Standardize booking channel labels.
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Clean booking channels:", sorted(df["booking_channel"].unique()))
print("Rows after standardizing channels:", len(df))

# (c) Parse dates and count missing values before dropping rows.
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("\nRows after parsing dates:", len(df))
print("Missing or unreadable appointment dates:", df["appt_date"].isna().sum())
print("Missing statuses:", df["status"].isna().sum())

# (d) Remove bookings that lack an appointment date or outcome.
df = df.dropna(subset=["appt_date", "status"]).copy()
print("Rows after dropping missing dates or statuses:", len(df))

Raw shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values by column:
appointment_id      0
patient_id          0

In [14]:
import numpy as np

# Task 1.3 — Replace invalid ages and fill gaps before computing the IQR.
invalid_age = df["age"].notna() & ~df["age"].between(1, 100)
df.loc[invalid_age, "age"] = np.nan

age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Invalid ages:", invalid_age.sum())
print("Missing ages before imputation:", df["age"].isna().sum())
print("Median age:", age_median)
print("Median distance:", distance_median)

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

for column in ["distance_km", "fee_inr"]:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    capped_count = ((df[column] < lower) | (df[column] > upper)).sum()
    print(f"{column}: Q1={q1}, Q3={q3}, fences=({lower}, {upper}), capped={capped_count}")
    df[column] = np.clip(df[column], lower, upper)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest cleaned fee:", df["fee_inr"].max())

# Task 1.4 — Build features and summarize them with NumPy.
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = df["status"].eq("No-Show").astype(int)

columns = ["age", "distance_km", "fee_inr", "lead_days"]
X = df[columns].to_numpy()
long_lead = X[:, 3] > 14
distance = X[:, 1]
distance_z = (distance - np.mean(distance)) / np.std(distance)

print("\nColumn means:", dict(zip(columns, np.round(X.mean(axis=0), 2))))
print("Lead-day range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean():.1%})")
print(f"Lead days > 14: {long_lead.sum()} ({long_lead.mean():.1%})")
print("Distance rows with |z| > 2:", (np.abs(distance_z) > 2).sum())

Invalid ages: 5
Missing ages before imputation: 17
Median age: 48.0
Median distance: 8.25
distance_km: Q1=4.625, Q3=13.0, fences=(-7.9375, 25.5625), capped=26
fee_inr: Q1=500.0, Q3=787.5, fences=(68.75, 1218.75), capped=3
Largest cleaned fee: 1219

Column means: {'age': np.float64(48.17), 'distance_km': np.float64(9.5), 'fee_inr': np.float64(635.71), 'lead_days': np.float64(11.82)}
Lead-day range: 1 to 45
No-shows: 218 of 990 (22.0%)
Lead days > 14: 296 (29.9%)
Distance rows with |z| > 2: 53


In [15]:
import requests

clinics = pd.read_csv("clinics.csv")


def fetch_temperature(clinic):
    params = {
        "latitude": clinic.latitude,
        "longitude": clinic.longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }

    for attempt in range(1, 3):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10,
            )
            if response.status_code != 200:
                raise ValueError(f"HTTP {response.status_code}")

            temperature = response.json()["hourly"]["temperature_2m"][0]
            if temperature is None:
                raise ValueError("First temperature reading is missing")

            print(f"{clinic.clinic_name}: HTTP 200, {temperature} °C")
            return temperature

        except (requests.exceptions.RequestException, ValueError, KeyError,
                IndexError, TypeError) as error:
            print(f"{clinic.clinic_name}, attempt {attempt}: {error}")

    return None


clinics["temp_c"] = [
    fetch_temperature(clinic) for clinic in clinics.itertuples(index=False)
]
clinics.to_csv("clinics_enriched.csv", index=False)
print("Clinics exported:", len(clinics))

Kothrud Family Clinic: HTTP 200, 19.2 °C
Hadapsar Care Centre: HTTP 200, 19.5 °C
Sitabuldi Health Point: HTTP 200, 18.6 °C
Malviya Nagar Clinic: HTTP 200, 14.4 °C
Vaishali Nagar Clinic: HTTP 200, 14.4 °C
Edappally Medical Centre: HTTP 200, 27.9 °C
Vijay Nagar Clinic: HTTP 200, 16.6 °C
Gomti Nagar Care Point: HTTP 200, 14.0 °C
Clinics exported: 8


In [16]:
export_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days",
    "no_show",
]

clean = df.sort_values("appointment_id")[export_columns].copy()

for column in ["booked_on", "appt_date"]:
    clean[column] = clean[column].dt.strftime("%Y-%m-%d")

unique_ids = clean["appointment_id"].is_unique
missing_cells = int(clean.isna().sum().sum())
print("Unique appointment IDs:", unique_ids)
print("Missing cells:", missing_cells)

assert unique_ids, "Duplicate appointment IDs remain"
assert missing_cells == 0, "Missing values remain"

clean.to_csv("appointments_clean.csv", index=False)

with open("load_data.sql", "w", encoding="utf-8") as sql_file:
    for clinic in clinics.itertuples(index=False):
        temp_sql = "NULL" if pd.isna(clinic.temp_c) else str(clinic.temp_c)
        sql_file.write(
            f"INSERT INTO clinics VALUES ("
            f"'{clinic.clinic_id}', '{clinic.clinic_name}', '{clinic.city}', "
            f"'{clinic.tier}', {clinic.latitude}, {clinic.longitude}, {temp_sql});\n"
        )

    for row in clean.itertuples(index=False):
        sql_file.write(
            f"INSERT INTO appointments VALUES ("
            f"'{row.appointment_id}', '{row.patient_id}', '{row.clinic_id}', "
            f"'{row.booked_on}', '{row.appt_date}', {row.age}, '{row.gender}', "
            f"{row.distance_km}, {row.fee_inr}, '{row.booking_channel}', "
            f"'{row.visit_type}', '{row.reminder_sent}', {row.prior_no_shows}, "
            f"'{row.status}', {row.lead_days}, {row.no_show});\n"
        )

with open("load_data.sql", encoding="utf-8") as sql_file:
    insert_count = sum(line.startswith("INSERT INTO ") for line in sql_file)

print("Clean CSV shape:", clean.shape)
print("SQL INSERT statements:", insert_count)

Unique appointment IDs: True
Missing cells: 0
Clean CSV shape: (990, 16)
SQL INSERT statements: 998
